# Painel de acessibilidade universitária

**Pesquisa de percepção · 2026**

Visão geral das respostas sobre mobiliário, circulação e recursos visuais. O painel mantém contagens e percentuais por pergunta para facilitar a leitura da amostra.

A leitura é descritiva: cada indicador usa as respostas válidas da respectiva pergunta. As faixas agrupadas destacam níveis de atenção sem substituir as distribuições completas. Como a pesquisa não é uma série temporal, o painel não projeta tendências.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def localizar_arquivo_dados():
    diretorios = [Path.cwd(), Path("/content")]
    padrao = "Pesquisa_Acessibilidade_Universidade_2026 - Respostas ao formul* 1.csv"

    for diretorio in diretorios:
        if not diretorio.exists():
            continue
        arquivos = sorted(diretorio.glob(padrao))
        if arquivos:
            return arquivos[0]

    raise FileNotFoundError(
        "Não encontrei o CSV de respostas na pasta atual nem em /content. "
        "Coloque o arquivo Pesquisa_Acessibilidade_Universidade_2026 - Respostas ao formulário 1.csv "
        "na mesma pasta deste notebook ou em /content."
    )

def limpar_nome_coluna(nome):
    return re.sub(r"\s+", " ", str(nome)).strip()

CAMINHO_ARQUIVO = localizar_arquivo_dados()
dados = pd.read_csv(CAMINHO_ARQUIVO)
dados.columns = [limpar_nome_coluna(nome) for nome in dados.columns]

print(f"Arquivo carregado: {CAMINHO_ARQUIVO.name}")
print(f"Respostas recebidas: {len(dados)}")

In [ ]:
def obter_coluna(dados, numero_pergunta):
    prefixo = f"{numero_pergunta}."
    correspondencias = [coluna for coluna in dados.columns if coluna.startswith(prefixo)]
    if not correspondencias:
        raise KeyError(f"Não encontrei a pergunta {numero_pergunta} no CSV.")
    return correspondencias[0]

def normalizar_resposta(valor):
    if pd.isna(valor):
        return None
    texto = re.sub(r"\s+", " ", str(valor)).strip()
    return re.sub(r"^\d+\s+", "", texto)

def obter_respostas(dados, numero_pergunta):
    coluna = obter_coluna(dados, numero_pergunta)
    return dados[coluna].map(normalizar_resposta)

def calcular_contagem(respostas, categorias):
    respostas_validas = respostas.dropna()
    quantidade = int(respostas_validas.isin(categorias).sum())
    return quantidade, len(respostas_validas)

def calcular_distribuicao(respostas, ordem):
    respostas_validas = respostas.dropna()
    contagens = respostas_validas.value_counts()
    categorias = [
        {"categoria": categoria, "quantidade": int(contagens.get(categoria, 0))}
        for categoria in ordem
        if int(contagens.get(categoria, 0)) > 0
    ]
    categorias.extend(
        {"categoria": str(categoria), "quantidade": int(quantidade)}
        for categoria, quantidade in contagens.items()
        if categoria not in ordem
    )
    total = len(respostas_validas)
    for item in categorias:
        item["percentual"] = item["quantidade"] / total * 100 if total else 0
    return categorias

def classificar_identificacao(valor):
    if pd.isna(valor) or not str(valor).strip():
        return "Sem resposta"
    texto = str(valor)
    if re.search(r"\bPessoa (?:com|acima do peso)\b", texto, flags=re.IGNORECASE):
        return "Indicou um ou mais grupos"
    if "Nenhuma das alternativas" in texto:
        return "Nenhum grupo indicado"
    if "Prefiro não responder" in texto:
        return "Preferiu não responder"
    return "Outra resposta"

def resumir_grupos(dados):
    respostas = obter_respostas(dados, 4)
    classificacoes = respostas.map(classificar_identificacao)
    contagens = classificacoes.value_counts()
    total = len(classificacoes)
    quantidade = int(contagens.get("Indicou um ou mais grupos", 0))
    percentual = quantidade / total * 100 if total else 0
    return quantidade, total, percentual

def calcular_indicadores(dados):
    definicoes = [
        ("Desconforto físico ≥ às vezes", 7, ["Às vezes", "Frequentemente", "Sempre"],
         "Respondeu às vezes, frequentemente ou sempre."),
        ("Dificuldade para ler slides ≥ às vezes", 12, ["Às vezes", "Frequentemente", "Sempre"],
         "Respondeu às vezes, frequentemente ou sempre."),
        ("Cadeiras avaliadas como ruim ou muito ruim", 5, ["Ruim", "Muito ruim"],
         "Somou as avaliações ruim e muito ruim."),
        ("Rampas geram dificuldade ≥ às vezes", 10, ["Às vezes", "Frequentemente", "Sempre"],
         "Respondeu às vezes, frequentemente ou sempre."),
        ("Deslocamento entre andares difícil", 9, ["Difícil", "Muito difícil"],
         "Somou as avaliações difícil e muito difícil."),
        ("Elevadores: impacto médio ou muito alto", 8, ["Médio impacto", "Impacto muito alto"],
         "Somou impacto médio e muito alto.")
    ]
    indicadores = []
    for rotulo, numero_pergunta, categorias, definicao in definicoes:
        quantidade, total = calcular_contagem(obter_respostas(dados, numero_pergunta), categorias)
        percentual = quantidade / total * 100 if total else 0
        indicadores.append({
            "rotulo": rotulo,
            "numero": quantidade,
            "total": total,
            "percentual": percentual,
            "definicao": definicao
        })
    return sorted(indicadores, key=lambda item: item["percentual"], reverse=True)

def calcular_faixas(respostas, faixas):
    respostas_validas = respostas.dropna()
    total = len(respostas_validas)
    resultado = {}
    for nome_faixa, categorias in faixas.items():
        quantidade = int(respostas_validas.isin(categorias).sum())
        resultado[nome_faixa] = {
            "quantidade": quantidade,
            "total": total,
            "percentual": quantidade / total * 100 if total else 0
        }
    return resultado

In [ ]:
def criar_painel_acessibilidade(dados):
    cor_fundo = "#0E101B"
    cor_painel = "#191B2B"
    cor_borda = "#38334F"
    cor_texto = "#F7F4FC"
    cor_secundaria = "#C2BDD0"
    cor_roxa = "#9A70D6"
    cor_verde = "#35C58B"
    cor_amarela = "#F3BD55"
    cor_laranja = "#F28A55"
    cor_coral = "#EE7181"
    cores_faixas = [cor_verde, cor_amarela, cor_coral]

    figura = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=(
            "Sinais de atenção nas respostas",
            "Respostas por semestre",
            "Conforto das cadeiras",
            "Desconforto físico com mobiliário",
            "Deslocamento e circulação",
            "Leitura de slides e qualidade do projetor"
        ),
        vertical_spacing=0.14,
        horizontal_spacing=0.12,
        row_heights=[0.39, 0.30, 0.31]
    )

    indicadores = calcular_indicadores(dados)
    figura.add_trace(
        go.Bar(
            x=[item["percentual"] for item in indicadores],
            y=[item["rotulo"] for item in indicadores],
            orientation="h",
            marker_color=[
                cor_coral if item["percentual"] >= 60
                else cor_amarela if item["percentual"] >= 30
                else cor_verde
                for item in indicadores
            ],
            text=[
                f'{item["percentual"]:.0f}% · {item["numero"]}/{item["total"]}'
                for item in indicadores
            ],
            textposition="outside",
            customdata=[
                [item["numero"], item["total"], item["definicao"]]
                for item in indicadores
            ],
            hovertemplate=(
                "<b>%{y}</b><br>%{x:.1f}%<br>"
                "%{customdata[0]} de %{customdata[1]} respostas<br>"
                "%{customdata[2]}<extra></extra>"
            ),
            cliponaxis=False
        ),
        row=1, col=1
    )

    semestres = calcular_distribuicao(
        obter_respostas(dados, 1),
        ["1º ao 2º", "3º ao 4º", "5º ao 6º", "7º ao 8º", "Acima do 8º"]
    )
    figura.add_trace(
        go.Bar(
            x=[item["quantidade"] for item in semestres],
            y=[item["categoria"] for item in semestres],
            orientation="h",
            marker_color=cor_roxa,
            text=[
                f'{item["quantidade"]} · {item["percentual"]:.0f}%'
                for item in semestres
            ],
            textposition="outside",
            customdata=[[item["percentual"]] for item in semestres],
            hovertemplate="<b>%{y}</b><br>%{x} respostas (%{customdata[0]:.1f}%)<extra></extra>",
            cliponaxis=False
        ),
        row=1, col=2
    )

    respostas_cadeiras = obter_respostas(dados, 5)
    conforto = calcular_distribuicao(
        respostas_cadeiras,
        ["Muito ruim", "Ruim", "Regular", "Boa", "Muito boa"]
    )
    figura.add_trace(
        go.Bar(
            x=[item["percentual"] for item in conforto],
            y=[item["categoria"] for item in conforto],
            orientation="h",
            marker_color=[
                cor_coral if item["categoria"] == "Muito ruim"
                else cor_laranja if item["categoria"] == "Ruim"
                else cor_amarela if item["categoria"] == "Regular"
                else cor_verde
                for item in conforto
            ],
            text=[f'{item["quantidade"]} · {item["percentual"]:.0f}%' for item in conforto],
            textposition="outside",
            customdata=[[item["quantidade"], len(respostas_cadeiras.dropna())] for item in conforto],
            hovertemplate="<b>%{y}</b><br>%{x:.1f}%<br>%{customdata[0]} de %{customdata[1]} respostas<extra></extra>",
            cliponaxis=False
        ),
        row=2, col=1
    )

    respostas_desconforto = obter_respostas(dados, 7)
    desconforto = calcular_distribuicao(
        respostas_desconforto,
        ["Nunca", "Raramente", "Às vezes", "Frequentemente", "Sempre"]
    )
    cores_desconforto = {
        "Nunca": cor_verde,
        "Raramente": "#65CBA1",
        "Às vezes": cor_amarela,
        "Frequentemente": cor_laranja,
        "Sempre": cor_coral
    }
    figura.add_trace(
        go.Bar(
            x=[item["percentual"] for item in desconforto],
            y=[item["categoria"] for item in desconforto],
            orientation="h",
            marker_color=[cores_desconforto.get(item["categoria"], cor_roxa) for item in desconforto],
            text=[f'{item["quantidade"]} · {item["percentual"]:.0f}%' for item in desconforto],
            textposition="outside",
            customdata=[[item["quantidade"], len(respostas_desconforto.dropna())] for item in desconforto],
            hovertemplate="<b>%{y}</b><br>%{x:.1f}%<br>%{customdata[0]} de %{customdata[1]} respostas<extra></extra>",
            cliponaxis=False
        ),
        row=2, col=2
    )

    faixas_mobilidade = {
        "Sem barreira / baixo impacto": ["Nenhum impacto", "Baixo impacto", "Fácil", "Muito fácil", "Nunca", "Raramente"],
        "Atenção / moderado": ["Médio impacto", "Regular", "Às vezes"],
        "Barreira elevada": ["Impacto muito alto", "Difícil", "Muito difícil", "Frequentemente", "Sempre"]
    }
    perguntas_mobilidade = [
        ("Impacto da ausência de elevadores", 8),
        ("Deslocamento entre andares", 9),
        ("Dificuldade com rampas", 10)
    ]
    resultados_mobilidade = [
        calcular_faixas(obter_respostas(dados, numero), faixas_mobilidade)
        for _, numero in perguntas_mobilidade
    ]
    for nome_faixa, cor in zip(faixas_mobilidade, cores_faixas):
        figura.add_trace(
            go.Bar(
                name=nome_faixa,
                x=[resultado[nome_faixa]["percentual"] for resultado in resultados_mobilidade],
                y=[rotulo for rotulo, _ in perguntas_mobilidade],
                orientation="h",
                marker_color=cor,
                text=[
                    f'{resultado[nome_faixa]["percentual"]:.0f}%'
                    if resultado[nome_faixa]["quantidade"] > 0 else ""
                    for resultado in resultados_mobilidade
                ],
                textposition="inside",
                customdata=[
                    [resultado[nome_faixa]["quantidade"], resultado[nome_faixa]["total"]]
                    for resultado in resultados_mobilidade
                ],
                hovertemplate=(
                    "<b>%{y}</b><br>" + nome_faixa + ": %{x:.1f}%"
                    "<br>%{customdata[0]} de %{customdata[1]} respostas<extra></extra>"
                ),
                showlegend=False
            ),
            row=3, col=1
        )

    faixas_visuais = {
        "Sem dificuldade / boa": ["Nunca", "Raramente", "Boa"],
        "Atenção / regular": ["Às vezes", "Regular"],
        "Dificuldade / ruim": ["Frequentemente", "Sempre", "Ruim", "Muito ruim"]
    }
    perguntas_visuais = [
        ("Leitura dos slides", 12),
        ("Qualidade do projetor", 13)
    ]
    resultados_visuais = [
        calcular_faixas(obter_respostas(dados, numero), faixas_visuais)
        for _, numero in perguntas_visuais
    ]
    for nome_faixa, cor in zip(faixas_visuais, cores_faixas):
        figura.add_trace(
            go.Bar(
                name=nome_faixa,
                x=[resultado[nome_faixa]["percentual"] for resultado in resultados_visuais],
                y=[rotulo for rotulo, _ in perguntas_visuais],
                orientation="h",
                marker_color=cor,
                text=[
                    f'{resultado[nome_faixa]["percentual"]:.0f}%'
                    if resultado[nome_faixa]["quantidade"] > 0 else ""
                    for resultado in resultados_visuais
                ],
                textposition="inside",
                customdata=[
                    [resultado[nome_faixa]["quantidade"], resultado[nome_faixa]["total"]]
                    for resultado in resultados_visuais
                ],
                hovertemplate=(
                    "<b>%{y}</b><br>" + nome_faixa + ": %{x:.1f}%"
                    "<br>%{customdata[0]} de %{customdata[1]} respostas<extra></extra>"
                ),
                showlegend=True
            ),
            row=3, col=2
        )

    figura.update_layout(
        height=1320,
        autosize=True,
        barmode="stack",
        bargap=0.34,
        paper_bgcolor=cor_fundo,
        plot_bgcolor=cor_painel,
        font=dict(family="Arial, sans-serif", color=cor_texto, size=12),
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=-0.035,
            yanchor="bottom",
            font=dict(size=10, color=cor_secundaria),
            bgcolor="rgba(0,0,0,0)"
        ),
        margin=dict(l=76, r=44, t=320, b=170),
        hoverlabel=dict(bgcolor="#26263A", font_color=cor_texto, bordercolor=cor_borda)
    )

    for numero_eixo in range(1, 7):
        sufixo = "" if numero_eixo == 1 else str(numero_eixo)
        eixo_x = figura.layout["xaxis" + sufixo]
        eixo_y = figura.layout["yaxis" + sufixo]
        dominio_x = eixo_x.domain
        dominio_y = eixo_y.domain
        figura.add_shape(
            type="rect",
            xref="paper",
            yref="paper",
            x0=max(0, dominio_x[0] - 0.025),
            x1=min(1, dominio_x[1] + 0.025),
            y0=max(0, dominio_y[0] - 0.035),
            y1=min(1, dominio_y[1] + 0.035),
            fillcolor=cor_painel,
            line=dict(color=cor_borda, width=1),
            layer="below"
        )
        linha = (numero_eixo - 1) // 2 + 1
        coluna = (numero_eixo - 1) % 2 + 1
        figura.update_xaxes(
            showgrid=True,
            gridcolor="#303247",
            zeroline=False,
            tickfont=dict(color=cor_secundaria, size=10),
            title_font=dict(color=cor_secundaria, size=10),
            row=linha,
            col=coluna
        )
        figura.update_yaxes(
            showgrid=False,
            zeroline=False,
            autorange="reversed",
            automargin=True,
            tickfont=dict(color=cor_texto, size=10),
            row=linha,
            col=coluna
        )

    figura.update_xaxes(title_text="Respostas que atendem ao critério (%)", range=[0, 100], ticksuffix="%", row=1, col=1)
    figura.update_xaxes(title_text="Quantidade de respostas", rangemode="tozero", row=1, col=2)
    figura.update_xaxes(title_text="Percentual das respostas", range=[0, 100], ticksuffix="%", row=2, col=1)
    figura.update_xaxes(title_text="Percentual das respostas", range=[0, 100], ticksuffix="%", row=2, col=2)
    figura.update_xaxes(title_text="Distribuição das respostas (%)", range=[0, 100], ticksuffix="%", row=3, col=1)
    figura.update_xaxes(title_text="Distribuição das respostas (%)", range=[0, 100], ticksuffix="%", row=3, col=2)

    for anotacao in figura.layout.annotations:
        anotacao.font = dict(size=13, color=cor_texto, family="Arial, sans-serif")
        anotacao.align = "left"

    indicadores = calcular_indicadores(dados)
    quantidade_grupos, total_grupos, percentual_grupos = resumir_grupos(dados)
    desconforto_kpi = next(item for item in indicadores if item["rotulo"].startswith("Desconforto físico"))
    leitura_kpi = next(item for item in indicadores if item["rotulo"].startswith("Dificuldade para ler"))
    kpis = [
        ("RESPOSTAS RECEBIDAS", str(len(dados)), "formulários"),
        ("DESCONFORTO COM MOBILIÁRIO", f'{desconforto_kpi["percentual"]:.0f}%', f'{desconforto_kpi["numero"]} de {desconforto_kpi["total"]} · ao menos às vezes'),
        ("DIFICULDADE COM SLIDES", f'{leitura_kpi["percentual"]:.0f}%', f'{leitura_kpi["numero"]} de {leitura_kpi["total"]} · ao menos às vezes'),
        ("INDICARAM UM GRUPO", f"{percentual_grupos:.0f}%", f"{quantidade_grupos} de {total_grupos} respostas")
    ]

    figura.add_shape(
        type="rect", xref="paper", yref="paper",
        x0=0.02, x1=0.98, y0=1.25, y1=1.38,
        fillcolor="#4B3A69", line=dict(color="#735196", width=1), layer="below"
    )
    figura.add_annotation(
        x=0.5, y=1.335, xref="paper", yref="paper",
        text="<b>ACESSIBILIDADE NA UNIVERSIDADE</b>",
        align="center", showarrow=False, xanchor="center", yanchor="middle",
        font=dict(size=24, color=cor_texto, family="Arial, sans-serif")
    )
    figura.add_annotation(
        x=0.5, y=1.275, xref="paper", yref="paper",
        text="Pesquisa de percepção · 2026 · " + str(len(dados)) + " respostas",
        align="center", showarrow=False, xanchor="center", yanchor="middle",
        font=dict(size=13, color=cor_secundaria, family="Arial, sans-serif")
    )
    for indice, (rotulo, valor, detalhe) in enumerate(kpis):
        inicio = 0.025 + indice * 0.245
        figura.add_shape(
            type="rect",
            xref="paper",
            yref="paper",
            x0=inicio,
            x1=inicio + 0.225,
            y0=1.075,
            y1=1.205,
            fillcolor=cor_painel,
            line=dict(color=cor_borda, width=1),
            layer="above"
        )
        cor_valor = cor_roxa if indice in [0, 3] else cor_coral
        figura.add_annotation(
            x=inicio + 0.012,
            y=1.14,
            xref="paper",
            yref="paper",
            text=(
                f"<span style='font-size:10px;color:#C2BDD0'>{rotulo}</span>"
                f"<br><b style='font-size:22px;color:{cor_valor}'>{valor}</b>"
                f"<br><span style='font-size:10px;color:#C2BDD0'>{detalhe}</span>"
            ),
            align="left",
            showarrow=False,
            xanchor="left",
            yanchor="middle"
        )

    figura.add_annotation(
        x=0.03,
        y=-0.12,
        xref="paper",
        yref="paper",
        text=(
            "Faixas agrupadas: verde = sem dificuldade/baixo impacto; "
            "amarelo = atenção/moderado; coral = dificuldade/barreira elevada. "
            "A pergunta aberta 14 não recebeu respostas."
        ),
        align="left",
        showarrow=False,
        font=dict(size=10, color=cor_secundaria)
    )
    return figura

painel_acessibilidade = criar_painel_acessibilidade(dados)
painel_acessibilidade.show()

### Como ler os indicadores

- Os percentuais usam as respostas válidas de cada pergunta; passe o cursor sobre as barras para ver contagem, denominador e critério.
- As barras de circulação e recursos visuais agrupam respostas em três faixas para facilitar a comparação. As distribuições de conforto e desconforto permanecem detalhadas.
- As respostas de identificação foram agregadas em “indicou um ou mais grupos”; o painel não expõe categorias sensíveis com poucos casos.
- Os resultados descrevem apenas as respostas recebidas. A pergunta aberta de sugestões está vazia, então não há recomendações textuais a resumir.